In [1]:
import gymnasium as gym

In [2]:
env = gym.make("CartPole-v1")
obs, _ = env.reset()
print(obs.shape)  # state = 4 values

(4,)


In [3]:
def expert_policy(obs):
    angle = obs[2]
    return 1 if angle > 0 else 0

In [4]:
import numpy as np

states = []
actions = []

for episode in range(100):
    obs, _ = env.reset()
    done = False
    
    while not done:
        act = expert_policy(obs)
        
        states.append(obs)
        actions.append(act)
        
        obs, _, terminated, truncated, _ = env.step(act)
        done = terminated or truncated

states = np.array(states)
actions = np.array(actions)
#collecting demonstration 

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

#neural network policy
class BCPolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
        
    def forward(self, x):
        return self.net(x)

model = BCPolicy()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [6]:
X = torch.tensor(states, dtype=torch.float32)
y = torch.tensor(actions, dtype=torch.long)

# Training loop (Behavioral Cloning)
for epoch in range(20):
    logits = model(X)
    loss = loss_fn(logits, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(epoch, loss.item())

0 0.7000424861907959
1 0.6973500847816467
2 0.6949092745780945
3 0.6927181482315063
4 0.6907650232315063
5 0.6890274882316589
6 0.6874741315841675
7 0.6860703825950623
8 0.6847811341285706
9 0.6835746765136719
10 0.6824212074279785
11 0.6812978386878967
12 0.6801847219467163
13 0.6790696978569031
14 0.6779420971870422
15 0.6767967939376831
16 0.6756303310394287
17 0.6744430661201477
18 0.6732363104820251
19 0.6720114350318909


In [7]:
def run_policy(policy, episodes=10):
    rewards = []
    
    for _ in range(episodes):
        obs, _ = env.reset()
        done = False
        total = 0
        
        while not done:
            with torch.no_grad():
                act = policy(torch.tensor(obs, dtype=torch.float32))
                act = torch.argmax(act).item()
                
            obs, r, terminated, truncated, _ = env.step(act)
            done = terminated or truncated
            total += r
        
        rewards.append(total)
    
    return np.mean(rewards)

print("BC reward:", run_policy(model))

BC reward: 500.0


In [8]:
for dagger_iter in range(5):
    
    new_states = []
    new_actions = []
    
    obs, _ = env.reset()
    done = False
    
    while not done:
        # learner action
        with torch.no_grad():
            act = model(torch.tensor(obs, dtype=torch.float32))
            act = torch.argmax(act).item()
        
        # expert correction
        expert_act = expert_policy(obs)
        
        new_states.append(obs)
        new_actions.append(expert_act)
        
        obs, _, terminated, truncated, _ = env.step(act)
        done = terminated or truncated
    
    # aggregate dataset
    states = np.vstack([states, new_states])
    actions = np.hstack([actions, new_actions])
    
    # retrain
    X = torch.tensor(states, dtype=torch.float32)
    y = torch.tensor(actions, dtype=torch.long)
    
    for epoch in range(5):
        logits = model(X)
        loss = loss_fn(logits, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print("DAgger iter", dagger_iter, "reward:", run_policy(model))

DAgger iter 0 reward: 500.0
DAgger iter 1 reward: 500.0
DAgger iter 2 reward: 500.0
DAgger iter 3 reward: 459.1
DAgger iter 4 reward: 35.4


In [9]:
import time
import torch

def run_policy_visual(policy, episodes=3):
    env = gym.make("CartPole-v1", render_mode="human")
    
    for ep in range(episodes):
        obs, _ = env.reset()
        done = False
        
        while not done:
            with torch.no_grad():
                act = policy(torch.tensor(obs, dtype=torch.float32))
                act = torch.argmax(act).item()
            
            obs, _, terminated, truncated, _ = env.step(act)
            done = terminated or truncated
            
            time.sleep(0.02)  # slow for visibility
    
    env.close()

In [10]:
run_policy_visual(model)